# Balanced-Sampling XGBoost Experiment

This notebook trains a multiclass XGBoost classifier using class-aware sampling.

**Purpose:** reduce majority-class dominance and improve evaluation across minority attack classes, accepting lower overall accuracy as a tradeoff.

> Use only authorized data. The dataset and generated model artifacts are not included in this repository.


In [ ]:
import os
from pathlib import Path

import boto3
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

DATA_S3_URI = os.getenv("SECURITY_DATA_S3_URI", "s3://YOUR-BUCKET/analytics/")
MODEL_OUTPUT_DIR = Path(os.getenv("MODEL_OUTPUT_DIR", "../models"))
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "Dst Port",
    "Protocol",
    "Flow Duration",
    "Flow Byts/s",
    "Flow Pkts/s",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts",
]


## Load the network telemetry


In [ ]:
df = pd.read_parquet(DATA_S3_URI)

print("Rows and columns:", df.shape)
print(df["Label"].value_counts())


## Build a class-aware sample

The sampling caps are based on the original experiment. `min()` prevents failures when a class contains fewer rows than the requested cap. Classes not listed in `CAPS` are retained in full.


In [ ]:
CAPS = {
    "Benign": 500_000,
    "DDoS attacks-LOIC-HTTP": 200_000,
    "DDOS attack-HOIC": 200_000,
    "DoS attacks-Hulk": 200_000,
    "Bot": 200_000,
}

samples = []
for label, group in df.groupby("Label"):
    cap = CAPS.get(label)
    if cap is None:
        samples.append(group)
    else:
        samples.append(
            group.sample(
                n=min(cap, len(group)),
                random_state=42,
            )
        )

df_sample = pd.concat(samples, ignore_index=True)

print(df_sample["Label"].value_counts())
print("Sample rows:", len(df_sample))


## Prepare features and split the data


In [ ]:
X = df_sample[FEATURES]
encoder = LabelEncoder()
y = encoder.fit_transform(df_sample["Label"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Classes:", list(encoder.classes_))


## Train the balanced-sampling model


In [ ]:
balanced_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=len(encoder.classes_),
    random_state=42,
    n_jobs=-1,
)

balanced_model.fit(X_train, y_train)


## Evaluate performance

Compare macro F1 with the natural-distribution model. Macro F1 gives each class equal weight and is useful when minority attack classes matter.


In [ ]:
predictions = balanced_model.predict(X_test)

print(classification_report(
    y_test,
    predictions,
    target_names=encoder.classes_,
    zero_division=0,
))

print("Training accuracy:", balanced_model.score(X_train, y_train))
print("Test accuracy:", balanced_model.score(X_test, y_test))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=encoder.classes_,
    xticks_rotation=90,
)
plt.tight_layout()
plt.show()


## Inspect feature importance


In [ ]:
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": balanced_model.feature_importances_,
}).sort_values("importance", ascending=False)

importance


## Save trusted local artifacts

Use the label encoder from the same class set and training process as both dashboard models.


In [ ]:
joblib.dump(balanced_model, MODEL_OUTPUT_DIR / "xgboost_balanced.pkl")
joblib.dump(encoder, MODEL_OUTPUT_DIR / "label_encoder.pkl")

print("Saved artifacts to", MODEL_OUTPUT_DIR.resolve())


## Optional S3 upload


In [ ]:
# MODEL_BUCKET = os.getenv("MODEL_BUCKET")
# if MODEL_BUCKET:
#     s3 = boto3.client("s3")
#     s3.upload_file(
#         MODEL_OUTPUT_DIR / "xgboost_balanced.pkl",
#         MODEL_BUCKET,
#         "models/xgboost_balanced.pkl",
#     )
#     s3.upload_file(
#         MODEL_OUTPUT_DIR / "label_encoder.pkl",
#         MODEL_BUCKET,
#         "models/label_encoder.pkl",
#     )


## Interpretation

In the original experiment, balanced sampling lowered overall accuracy but improved macro F1. This is a meaningful security tradeoff: a model that performs well on the dominant class can still miss rare attacks. Further work should use temporal validation, cross-validation, calibrated probabilities, and explicit false-positive cost analysis.
